# CBP Run 2 — CFP Controlled Degradation Mechanism Test

This notebook tests a narrower question than a benchmark comparison:

> Does controlled acquisition degradation make CBP's evidence structure more informative than mean-only state summaries?

The first Run-2 conditions are frozen before inspection:

- `lowres_32`: downsample to a 32-pixel short side, then upsample to the original canvas.
- `blur_3`: Gaussian blur with radius 3.

The clean CFP result is rerun on the **same common sample set** as the degraded conditions. This avoids comparing different detected-image subsets if degradation causes face-detection failures.

Primary diagnostic remains **Gate A / E1**. Gate B is secondary because the clean CFP run was close to an accept-all regime.

Source code is cloned from GitHub. Raw CFP stays on Drive. Degraded images are temporary local files; only embedding caches and experiment reports persist on Drive.


In [ ]:
# ============================================================
# 0. RUN-2 CONFIG
# ============================================================

USE_GOOGLE_DRIVE = True

GITHUB_REPO_URL = "https://github.com/Tai12345-Ai/FIQA-CBP.git"
GITHUB_REF = "main"

CFP_ZIP_DRIVE = "/content/drive/MyDrive/datasets/CFP/cfp-dataset.zip"
CFP_LOCAL_DIR = "/content/CFP"
PROJECT_DIR = "/content/FIQA-CBP"

MODEL_NAME = "buffalo_l"

# Existing clean embedding cache from Run 1.
CLEAN_EMBEDDINGS_NPZ = "/content/drive/MyDrive/CBP/cache/cfp_buffalo_l_embeddings.npz"

# Predeclared Run-2 conditions. Do not tune these after inspecting outcomes.
RUN2_CONDITIONS = ["lowres_32", "blur_3"]

RUN2_CACHE_ROOT = "/content/drive/MyDrive/CBP/cache/run2_cfp_degradation"
RUN2_OUTPUT_ROOT = "/content/drive/MyDrive/CBP/runs/cfp_run2_degradation"

# Require most clean samples to survive every condition; otherwise the mechanism
# comparison becomes dominated by detector failures.
MIN_COMMON_FRACTION = 0.95

FORCE_REEXTRACT = False
FORCE_RERUN = False
FAST_MODE = False
SEED = 20261021


In [ ]:
# ============================================================
# 1. Mount Drive + clone source + install dependencies
# ============================================================

import os, sys, shutil, subprocess, json, time
from pathlib import Path

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

project_dir = Path(PROJECT_DIR)
if project_dir.exists():
    shutil.rmtree(project_dir)

subprocess.check_call([
    "git", "clone", "--depth", "1", "--branch", GITHUB_REF,
    GITHUB_REPO_URL, str(project_dir),
])

PROJECT_DIR = project_dir
os.chdir(PROJECT_DIR)

commit_sha = subprocess.check_output(
    ["git", "-C", str(PROJECT_DIR), "rev-parse", "HEAD"],
    text=True,
).strip()

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", ".[dev]"])
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "insightface>=0.7.3", "onnxruntime-gpu>=1.16",
])

import torch
import onnxruntime as ort

PROVIDER = (
    "CUDAExecutionProvider"
    if "CUDAExecutionProvider" in ort.get_available_providers()
    else "CPUExecutionProvider"
)

print("PROJECT_DIR :", PROJECT_DIR)
print("Commit SHA  :", commit_sha)
print("CUDA        :", torch.cuda.is_available())
print("ORT provider:", PROVIDER)


In [ ]:
# ============================================================
# 2. CFP preflight + clean manifest
# ============================================================

import zipfile
import pandas as pd
from cbp.data.manifest import FaceManifest

zip_path = Path(CFP_ZIP_DRIVE)
assert zip_path.exists(), f"Missing CFP ZIP: {zip_path}"

local_root = Path(CFP_LOCAL_DIR)
if not local_root.exists() or not any(local_root.iterdir()):
    local_root.mkdir(parents=True, exist_ok=True)
    print("Extracting CFP ZIP...")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(local_root)
else:
    print("Reusing existing CFP extraction:", local_root)

candidates = [
    local_root / "cfp-dataset" / "Data" / "Images",
    local_root / "Data" / "Images",
]
candidates += list(local_root.glob("*/cfp-dataset/Data/Images"))
candidates += list(local_root.glob("*/Data/Images"))

image_root = next((p for p in candidates if p.exists()), None)
assert image_root is not None, "Could not locate CFP Data/Images"

clean_manifest = FaceManifest.from_identity_folders(image_root)
manifest_path = PROJECT_DIR / "data" / "CFP_run2_clean_manifest.csv"
clean_manifest.to_csv(manifest_path)

df = clean_manifest.frame
print("Image root :", image_root)
print("Images     :", len(df))
print("Identities :", df["identity_id"].nunique())
print("Manifest   :", manifest_path)

assert len(df) >= 6000
assert df["identity_id"].nunique() >= 400


In [ ]:
# ============================================================
# 3. Build degraded images locally + extract frozen embeddings
# ============================================================

from cbp.data.degradation import build_degraded_manifest

cache_root = Path(RUN2_CACHE_ROOT)
cache_root.mkdir(parents=True, exist_ok=True)

degraded_embedding_paths = {}
degradation_diagnostics = {}

for condition in RUN2_CONDITIONS:
    emb_path = cache_root / f"cfp_buffalo_l_{condition}.npz"
    degraded_embedding_paths[condition] = emb_path

    if emb_path.exists() and not FORCE_REEXTRACT:
        print(f"[{condition}] reuse embedding cache:", emb_path)
    else:
        local_condition_root = Path("/content/CFP_run2_degraded") / condition
        if local_condition_root.exists():
            shutil.rmtree(local_condition_root)

        print(f"[{condition}] materializing degraded images...")
        t0 = time.time()
        degraded_manifest = build_degraded_manifest(
            clean_manifest,
            output_root=local_condition_root,
            condition=condition,
        )
        degraded_manifest_path = PROJECT_DIR / "data" / f"CFP_run2_{condition}_manifest.csv"
        degraded_manifest.to_csv(degraded_manifest_path)
        print(f"[{condition}] generated {len(degraded_manifest.frame)} images in {(time.time()-t0)/60:.1f} min")

        metadata_path = Path(str(emb_path) + ".metadata.json")
        failure_path = Path(str(emb_path) + ".failures.tsv")

        print(f"[{condition}] extracting frozen {MODEL_NAME} embeddings...")
        subprocess.check_call([
            sys.executable, "-m", "scripts.extract_embeddings",
            "--manifest", str(degraded_manifest_path),
            "--output", str(emb_path),
            "--model", MODEL_NAME,
            "--provider", PROVIDER,
            "--ctx-id", "0" if PROVIDER == "CUDAExecutionProvider" else "-1",
            "--metadata", str(metadata_path),
            "--failure-log", str(failure_path),
        ], cwd=PROJECT_DIR)

        # Degraded images are disposable; persistent artifacts are embeddings/reports.
        shutil.rmtree(local_condition_root, ignore_errors=True)

    metadata_path = Path(str(emb_path) + ".metadata.json")
    if metadata_path.exists():
        meta = json.loads(metadata_path.read_text(encoding="utf-8"))
        degradation_diagnostics[condition] = {
            "samples_succeeded": meta.get("samples_succeeded"),
            "samples_failed": meta.get("samples_failed"),
            "embedding_dim": meta.get("embedding_dim"),
        }
    else:
        degradation_diagnostics[condition] = {"metadata": "missing"}

print(json.dumps(degradation_diagnostics, indent=2))


In [ ]:
# ============================================================
# 4. Pairwise-align clean vs each degradation condition
# ============================================================

import numpy as np
from cbp.fr.embeddings import EmbeddingTable

clean_path = Path(CLEAN_EMBEDDINGS_NPZ)
assert clean_path.exists(), f"Missing clean embedding cache: {clean_path}"

clean_table = EmbeddingTable.load(clean_path)
clean_ids = clean_table.sample_ids.tolist()
clean_id_set = set(clean_ids)
clean_index = {sid: i for i, sid in enumerate(clean_ids)}

aligned_root = cache_root / "aligned_pairwise"
aligned_root.mkdir(parents=True, exist_ok=True)

pairwise_paths = {}
alignment_rows = []

print("Clean samples:", len(clean_ids))

for condition, path in degraded_embedding_paths.items():
    degraded = EmbeddingTable.load(path)
    degraded_ids = degraded.sample_ids.tolist()
    degraded_id_set = set(degraded_ids)

    common_order = [sid for sid in clean_ids if sid in degraded_id_set]
    pair_fraction = len(common_order) / len(clean_ids)

    print(
        f"{condition:12s} degraded={len(degraded_ids):5d} "
        f"pairwise_common={len(common_order):5d} "
        f"fraction={pair_fraction:.3%}"
    )

    alignment_rows.append({
        "condition": condition,
        "clean_samples": len(clean_ids),
        "degraded_samples": len(degraded_ids),
        "pairwise_common_samples": len(common_order),
        "pairwise_common_fraction": pair_fraction,
        "valid_for_mechanism_test": pair_fraction >= MIN_COMMON_FRACTION,
    })

    if pair_fraction < MIN_COMMON_FRACTION:
        print(
            f"  SKIP {condition}: detector survival {pair_fraction:.3%} "
            f"< required {MIN_COMMON_FRACTION:.1%}. "
            "Treat this as detector-failure-dominated stress, not Gate-A evidence."
        )
        continue

    degraded_index = {sid: i for i, sid in enumerate(degraded_ids)}
    clean_indices = np.asarray([clean_index[sid] for sid in common_order], dtype=np.int64)
    degraded_indices = np.asarray([degraded_index[sid] for sid in common_order], dtype=np.int64)

    clean_aligned = clean_table.subset(clean_indices)
    degraded_aligned = degraded.subset(degraded_indices)

    assert clean_aligned.sample_ids.tolist() == common_order
    assert degraded_aligned.sample_ids.tolist() == common_order
    assert clean_aligned.identity_ids.tolist() == degraded_aligned.identity_ids.tolist()

    clean_out = aligned_root / f"clean_for_{condition}.npz"
    degraded_out = aligned_root / f"{condition}.npz"

    clean_aligned.save(clean_out)
    degraded_aligned.save(degraded_out)

    pairwise_paths[condition] = {
        "clean": clean_out,
        "degraded": degraded_out,
        "common_fraction": pair_fraction,
        "common_samples": len(common_order),
    }

alignment_df = pd.DataFrame(alignment_rows)
display(alignment_df)

valid_conditions = list(pairwise_paths.keys())
print("Valid conditions:", valid_conditions)

alignment_report = {
    "clean_samples": len(clean_ids),
    "minimum_pairwise_common_fraction": MIN_COMMON_FRACTION,
    "valid_conditions": valid_conditions,
    "rows": alignment_df.to_dict(orient="records"),
    "degradation_diagnostics": degradation_diagnostics,
}
(cache_root / "alignment_report.json").write_text(
    json.dumps(alignment_report, indent=2),
    encoding="utf-8",
)

if not valid_conditions:
    raise RuntimeError(
        "No degradation condition retained enough detected samples for a fair "
        "pairwise mechanism test. Do not lower MIN_COMMON_FRACTION post hoc; "
        "define a separate milder Run-2b detector-viability study instead."
    )


In [ ]:
# ============================================================
# 5. Run the same CBP protocol on each pairwise-matched clean/degraded pair
# ============================================================

import yaml
from cbp.experiment import run_experiment

base_cfg = yaml.safe_load(
    (PROJECT_DIR / "configs" / "pilot.yaml").read_text(encoding="utf-8")
)

# Keep the Run-1 core design fixed.
base_cfg["seed"] = SEED
base_cfg["run_continuous_baseline"] = False
base_cfg["run_image_neural_baseline"] = False
base_cfg["certify_candidate_family"] = False
base_cfg["k_candidates"] = [4, 8, 16]
base_cfg["n_states"] = 64
base_cfg["spectral_dim"] = 8

if FAST_MODE:
    base_cfg["max_identities_per_split"] = min(base_cfg.get("max_identities_per_split") or 80, 80)
    base_cfg["router_epochs"] = 20
    base_cfg["router_patience"] = 5
    base_cfg["phenomenon_permutations"] = 100
    base_cfg["bootstrap_repetitions"] = 100
else:
    base_cfg["phenomenon_permutations"] = 1000
    base_cfg["bootstrap_repetitions"] = 1000

output_root = Path(RUN2_OUTPUT_ROOT)
output_root.mkdir(parents=True, exist_ok=True)

reports = {}

for condition in valid_conditions:
    pair = pairwise_paths[condition]
    reports[condition] = {}

    for variant, embedding_path in [
        ("clean", pair["clean"]),
        ("degraded", pair["degraded"]),
    ]:
        run_name = f"{condition}__{variant}"
        out_dir = output_root / run_name
        report_path = out_dir / "report.json"

        if report_path.exists() and not FORCE_RERUN:
            print(f"[{run_name}] reusing report:", report_path)
            reports[condition][variant] = json.loads(
                report_path.read_text(encoding="utf-8")
            )
            continue

        cfg = dict(base_cfg)
        cfg["embedding_npz"] = str(embedding_path)
        cfg["output_dir"] = str(out_dir)

        cfg_path = PROJECT_DIR / "configs" / f"run2_{run_name}.yaml"
        cfg_path.write_text(
            yaml.safe_dump(cfg, sort_keys=False),
            encoding="utf-8",
        )

        print("\n" + "=" * 72)
        print("PAIR CONDITION:", condition)
        print("VARIANT       :", variant)
        print("Embeddings    :", embedding_path)
        print("Output        :", out_dir)

        t0 = time.time()
        reports[condition][variant] = run_experiment(cfg_path)
        print(f"[{run_name}] finished in {(time.time()-t0)/60:.1f} min")


In [ ]:
# ============================================================
# 6. Run-2 pairwise mechanism summary
# ============================================================

rows = []

for condition in valid_conditions:
    for variant in ("clean", "degraded"):
        report = reports[condition][variant]
        gate_a = report["gates"]["A_phenomenon"]
        gate_b = report["gates"]["B_decision_relevance"]

        for context, a in gate_a["contexts"].items():
            b = gate_b["contexts"][context]
            rows.append({
                "condition": condition,
                "variant": variant,
                "context": context,
                "gate_A_pass": a["pass"],
                "evidence_excess_ratio": a["evidence_excess_ratio_over_permuted"],
                "mean_match_distance": a["mean_match_distance"],
                "n_pairs": a["n_pairs"],
                "descriptive_p": a["descriptive_permutation_p"],
                "gate_B_pass_current_rule": b["pass"],
                "coverage": b["coverage"],
                "fmr": b["fmr"],
                "genuine_yield": b["genuine_yield"],
                "yield_gain_vs_mean_only": b["yield_gain_over_matched_mean_only"],
                "pairwise_common_fraction": pairwise_paths[condition]["common_fraction"],
            })

summary_df = pd.DataFrame(rows)

clean_lookup = (
    summary_df[summary_df["variant"] == "clean"]
    .set_index(["condition", "context"])["evidence_excess_ratio"]
)

summary_df["delta_evidence_vs_pairwise_clean"] = summary_df.apply(
    lambda row: (
        row["evidence_excess_ratio"]
        - clean_lookup[(row["condition"], row["context"])]
        if row["variant"] == "degraded"
        else 0.0
    ),
    axis=1,
)

display(
    summary_df.sort_values(["condition", "context", "variant"])
    .reset_index(drop=True)
)

print("\n=== RUN-2 INTERPRETATION ===")
for condition in valid_conditions:
    block = summary_df[
        (summary_df["condition"] == condition)
        & (summary_df["variant"] == "degraded")
    ]
    improved = (block["delta_evidence_vs_pairwise_clean"] > 0).sum()
    passed = block["gate_A_pass"].sum()
    print(
        f"{condition}: Gate-A contexts passed={passed}/{len(block)}, "
        f"evidence-ratio improved vs pairwise clean={improved}/{len(block)}, "
        f"survival={pairwise_paths[condition]['common_fraction']:.1%}"
    )

invalid_conditions = [
    row["condition"]
    for row in alignment_rows
    if not row["valid_for_mechanism_test"]
]
if invalid_conditions:
    print(
        "\nExcluded from Gate-A mechanism inference because detector survival "
        "was below the predeclared threshold:",
        invalid_conditions,
    )

print("\nPrimary question:")
print(
    "Within the exact same detected sample set, does degradation increase "
    "held-out evidence structure beyond mean-only summaries?"
)
print(
    "Do not reinterpret Gate-B numerical PASS as practical superiority "
    "when yield gain is approximately zero."
)


In [ ]:
# ============================================================
# 7. Save compact Run-2 summary to Drive
# ============================================================

summary_path = Path(RUN2_OUTPUT_ROOT) / "run2_summary.csv"
summary_df.to_csv(summary_path, index=False)

summary_json = {
    "source_commit": commit_sha,
    "minimum_pairwise_common_fraction": MIN_COMMON_FRACTION,
    "valid_conditions": valid_conditions,
    "alignment": alignment_rows,
    "rows": summary_df.to_dict(orient="records"),
}
(Path(RUN2_OUTPUT_ROOT) / "run2_summary.json").write_text(
    json.dumps(summary_json, indent=2),
    encoding="utf-8",
)

print("Saved:", summary_path)
print("Saved:", Path(RUN2_OUTPUT_ROOT) / "run2_summary.json")
